# 3. Date & Time Cleaning and Feature Extraction

This notebook covers:
1. Parsing inconsistent date string formats using `pd.to_datetime()`.
2. Handling invalid timestamps safely with `errors='coerce'`.
3. Extracting tabular time features (year, month, day, day of week, weekend indicator) for ML models.
4. Calculating time elapsed (durations).

In [2]:
import numpy as np
import pandas as pd

# Raw dataset with inconsistent datetime formats and invalid entries
data = {
    'Transaction_ID': [101, 102, 103, 104, 105],
    'Raw_Date': ['2023-01-15', '15/02/2023', 'March 10, 2023', 'invalid_date', '2023-04-20 14:30:00'],
    'Signup_Date': ['2022-01-01', '2022-05-15', '2021-11-20', '2022-08-10', '2023-01-01']
}

df = pd.DataFrame(data)
print("=== RAW DATASET ===")
display(df)
print("\nInitial Data Types:")
print(df.dtypes)

=== RAW DATASET ===


,Transaction_ID,Raw_Date,Signup_Date
0,101,2023-01-15,2022-01-01
1,102,15/02/2023,2022-05-15
2,103,"March 10, 2023",2021-11-20
3,104,invalid_date,2022-08-10
4,105,2023-04-20 14:30:00,2023-01-01



Initial Data Types:
Transaction_ID     int64
Raw_Date          object
Signup_Date       object
dtype: object


---
## Part 1: Parsing Inconsistent Datetimes Safely

- `pd.to_datetime(..., format='mixed')` automatically reconciles mixed date formats.
- `errors='coerce'` converts unparseable strings (like `'invalid_date'`) into `NaT` (Not a Time) without crashing.

In [3]:
# Convert mixed string formats into standardized datetime objects
df['Clean_Date'] = pd.to_datetime(df['Raw_Date'], errors='coerce', format='mixed')
df['Signup_Date'] = pd.to_datetime(df['Signup_Date'], errors='coerce')

print("=== PARSED DATETIMES ===")
display(df[['Transaction_ID', 'Raw_Date', 'Clean_Date', 'Signup_Date']])
print("\nUpdated Data Types:")
print(df.dtypes)

=== PARSED DATETIMES ===


,Transaction_ID,Raw_Date,Clean_Date,Signup_Date
0,101,2023-01-15,2023-01-15 00:00:00,2022-01-01
1,102,15/02/2023,2023-02-15 00:00:00,2022-05-15
2,103,"March 10, 2023",2023-03-10 00:00:00,2021-11-20
3,104,invalid_date,NaT,2022-08-10
4,105,2023-04-20 14:30:00,2023-04-20 14:30:00,2023-01-01



Updated Data Types:
Transaction_ID             int64
Raw_Date                  object
Signup_Date       datetime64[ns]
Clean_Date        datetime64[ns]
dtype: object


---
## Part 2: Extracting Features for Machine Learning

ML models cannot process raw datetime objects directly. You must decompose them into numeric features:
- **Year, Month, Day**
- **Day of Week** (`0` = Monday, `6` = Sunday)
- **Is_Weekend** (Binary flag)
- **Account Age / Duration** (Time difference in days)

In [4]:
# 1. Extract component features
df['Year'] = df['Clean_Date'].dt.year
df['Month'] = df['Clean_Date'].dt.month
df['Day'] = df['Clean_Date'].dt.day
df['Day_Name'] = df['Clean_Date'].dt.day_name()
df['Is_Weekend'] = df['Clean_Date'].dt.dayofweek.isin([5, 6]).astype(int)

# 2. Compute duration in days between two datetime columns
df['Days_Since_Signup'] = (df['Clean_Date'] - df['Signup_Date']).dt.days

print("=== FINAL ML-READY DATETIME FEATURES ===")
display(df)

=== FINAL ML-READY DATETIME FEATURES ===


,Transaction_ID,Raw_Date,Signup_Date,Clean_Date,Year,Month,Day,Day_Name,Is_Weekend,Days_Since_Signup
0,101,2023-01-15,2022-01-01,2023-01-15 00:00:00,2023.0,1.0,15.0,Sunday,1,379.0
1,102,15/02/2023,2022-05-15,2023-02-15 00:00:00,2023.0,2.0,15.0,Wednesday,0,276.0
2,103,"March 10, 2023",2021-11-20,2023-03-10 00:00:00,2023.0,3.0,10.0,Friday,0,475.0
3,104,invalid_date,2022-08-10,NaT,NaN,NaN,NaN,NaN,0,NaN
4,105,2023-04-20 14:30:00,2023-01-01,2023-04-20 14:30:00,2023.0,4.0,20.0,Thursday,0,109.0
